# ========= SECTION 1: DATA LOADING & PREPROCESSING (with MD Column Alignment) =========

In [ ]:
# ========= SECTION 1: DATA LOADING & PREPROCESSING (with MD Column Alignment) =========

import os, sys, json, ast, time, copy, random, pickle, logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score


# ---- GLOBAL SETTINGS ----
SEED          = 42
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
KEY_COL       = "SMILES"
LABEL_COL     = "Activity"
OUT_ROOT      = r"D:\QSAR Cardiotoxicity\Binary_Multitask"
NUM_WORKERS   = 0
PIN_MEMORY    = (DEVICE == "cuda")   # True hanya jika GPU tersedia
NON_BLOCKING  = True
CV_FOLDS      = 10
DROPOUT       = 0.2
ALL_ENDPOINTS = ["hERG", "Cav1.2", "Nav1.5"]

os.makedirs(OUT_ROOT, exist_ok=True)
print(f"[INFO] DEVICE={DEVICE} | PIN_MEMORY={PIN_MEMORY}")


# ---- PATHS ----
FP_PATHS = {
    "Nav1.5": r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\Fingerprint\Nav1.5_DevSet_with_ECFP4_MACCS_APF.csv",
    "Cav1.2": r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\Fingerprint\Cav1.2_DevSet_with_ECFP4_MACCS_APF.csv",
    "hERG":   r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\Fingerprint\hERG_DevSet_with_ECFP4_MACCS_APF.csv",
}

GRAPH_PATHS = {
    "Nav1.5": r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\Graph\Nav1.5_DevSet_graphs_67feat_with_labels.pkl",
    "Cav1.2": r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\Graph\Cav1.2_DevSet_graphs_67feat_with_labels.pkl",
    "hERG":   r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\Graph\hERG_DevSet_graphs_67feat_with_labels.pkl",
}

MORDRED_PATHS = {
    "Nav1.5": r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\PP\Nav1.5_DevSet_RDKit_CDK.xlsx",
    "Cav1.2": r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\PP\Cav1.2_DevSet_RDKit_CDK.xlsx",
    "hERG":   r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\PP\hERG_DevSet_RDKit_CDK.xlsx",
}

META_MD_COLS = [
    "SMILES", "Activity", "pIC50", "Source",
    "Activity_Label", "Compound_ID", "InChIKey", "Max_Tanimoto_to_Dev",
]


# ---- HELPERS ----
def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark     = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)


def fast_parse_list(s):
    if isinstance(s, list):       return s
    if isinstance(s, np.ndarray): return s.tolist()
    if not isinstance(s, str):    raise TypeError(f"Unexpected type: {type(s)}")
    try:    return json.loads(s)
    except: return ast.literal_eval(s)


def compute_fp_matrix_from_df(df_fp, ec_col="ECFP4", mc_col="MACCS"):
    ec = df_fp[ec_col].tolist(); mc = df_fp[mc_col].tolist()
    ec0 = fast_parse_list(ec[0]); mc0 = fast_parse_list(mc[0])
    d_ec, d_mc = len(ec0), len(mc0)
    X = np.zeros((len(df_fp), d_ec + d_mc), dtype=np.float32)
    for i in range(len(df_fp)):
        e = np.asarray(fast_parse_list(ec[i]), dtype=np.float32)
        m = np.asarray(fast_parse_list(mc[i]), dtype=np.float32)
        X[i, :d_ec] = e; X[i, d_ec:] = m
    return X, d_ec, d_mc


def get_mordred_feature_cols(df_md, meta_cols):
    return [c for c in df_md.columns
            if c not in meta_cols and pd.api.types.is_numeric_dtype(df_md[c])]


def load_fp_df(path):
    df = pd.read_csv(path)
    print(f"[FP]    Loaded {path}  shape={df.shape}")
    return df


def load_mordred_df(path):
    df = pd.read_excel(path)
    print(f"[MD]    Loaded {path}  shape={df.shape}")
    return df


def load_graph_pkl(path):
    with open(path, "rb") as f: obj = pickle.load(f)
    print(f"[Graph] Loaded {path}  n={len(obj)}")
    return obj


# ================================================================
# MD Column Alignment
# ================================================================
def align_mordred_cols(raw_mordred_dict, meta_cols=META_MD_COLS, strategy="intersection"):
    """
    Align kolom fitur MD antar semua endpoint agar dimensi konsisten.

    Parameters
    ----------
    raw_mordred_dict : dict {endpoint: pd.DataFrame}
    meta_cols        : list kolom non-fitur yang diabaikan
    strategy         : "intersection" — kolom yang ada di SEMUA endpoint (aman, tidak ada kolom nol palsu)
                       "union"        — semua kolom, endpoint yang tidak punya diisi 0

    Returns
    -------
    aligned_dict     : dict {endpoint: pd.DataFrame}
    shared_feat_cols : list[str]
    """
    feat_cols_per_ep = {}
    for ep, df in raw_mordred_dict.items():
        feat_cols_per_ep[ep] = set(get_mordred_feature_cols(df, meta_cols))
        print(f"[MD Align] {ep}: {len(feat_cols_per_ep[ep])} feature cols")

    if strategy == "intersection":
        shared = set.intersection(*feat_cols_per_ep.values())
        shared_feat_cols = sorted(list(shared))
        print(f"\n[MD Align] Strategy=INTERSECTION → {len(shared_feat_cols)} shared feature cols")
        for ep, cols in feat_cols_per_ep.items():
            dropped = cols - shared
            if dropped:
                print(f"  ⚠️  {ep}: {len(dropped)} cols DROPPED (tidak ada di semua endpoint)")

    elif strategy == "union":
        shared = set.union(*feat_cols_per_ep.values())
        shared_feat_cols = sorted(list(shared))
        print(f"\n[MD Align] Strategy=UNION → {len(shared_feat_cols)} feature cols (kolom tidak ada → diisi 0)")

    else:
        raise ValueError(f"Unknown strategy: '{strategy}'. Pilih 'intersection' atau 'union'.")

    aligned_dict = {}
    for ep, df in raw_mordred_dict.items():
        meta_present = [c for c in meta_cols if c in df.columns]
        df_aligned   = df[meta_present].copy()

        for col in shared_feat_cols:
            if col in df.columns:
                df_aligned[col] = df[col].values
            else:
                df_aligned[col] = 0.0
                print(f"  [MD Align] {ep}: col '{col}' tidak ada → diisi 0")

        df_aligned[shared_feat_cols] = df_aligned[shared_feat_cols].apply(
            pd.to_numeric, errors="coerce"
        ).fillna(0.0)

        aligned_dict[ep] = df_aligned
        print(f"[MD Align] {ep}: aligned shape={df_aligned.shape}  feat_dim={len(shared_feat_cols)}")

    # Verifikasi dimensi — pakai df_aligned langsung, bukan df.columns
    dims = {ep: df_al[shared_feat_cols].shape[1] for ep, df_al in aligned_dict.items()}
    assert len(set(dims.values())) == 1, f"❌ Dimensi MD tidak konsisten setelah alignment: {dims}"
    print(f"\n✅ MD Alignment DONE — semua endpoint punya {len(shared_feat_cols)} fitur MD identik.")

    log_path = os.path.join(OUT_ROOT, "md_aligned_feature_cols.txt")
    with open(log_path, "w") as f:
        f.write(f"Strategy: {strategy}\n")
        f.write(f"N features: {len(shared_feat_cols)}\n\n")
        for col in shared_feat_cols:
            f.write(col + "\n")
    print(f"[MD Align] Feature col list saved → {log_path}")

    return aligned_dict, shared_feat_cols


# ---- LOAD RAW DATA ----
RAW_FP      = {}
RAW_MORDRED = {}
RAW_GRAPH   = {}

for endpoint in ALL_ENDPOINTS:
    RAW_FP[endpoint]      = load_fp_df(FP_PATHS[endpoint])
    RAW_MORDRED[endpoint] = load_mordred_df(MORDRED_PATHS[endpoint])
    g_path = GRAPH_PATHS[endpoint]
    RAW_GRAPH[endpoint]   = load_graph_pkl(g_path) if os.path.exists(g_path) else None


# ================================================================
# Jalankan MD Alignment SEBELUM build CORE_UNI / CORE_MULTI
# ================================================================
RAW_MORDRED_ALIGNED, SHARED_MD_COLS = align_mordred_cols(
    RAW_MORDRED,
    meta_cols=META_MD_COLS,
    strategy="intersection",   # ganti ke "union" jika ingin pakai semua kolom
)

print(f"\n[INFO] Jumlah fitur MD setelah alignment : {len(SHARED_MD_COLS)}")
print(f"[INFO] Contoh 5 kolom pertama            : {SHARED_MD_COLS[:5]}")

# ========= SECTION 1B: BUILD CORE_UNI & CORE_MULTI PER ENDPOINT =========

In [ ]:
# ========= SECTION 1B: BUILD CORE_UNI & CORE_MULTI PER ENDPOINT =========

CORE_UNI   = {}
CORE_MULTI = {}

for endpoint in ALL_ENDPOINTS:
    print(f"\n### Building CORE for {endpoint} ###")
    df_fp = RAW_FP[endpoint]
    df_md = RAW_MORDRED_ALIGNED[endpoint]
    g_obj = RAW_GRAPH[endpoint]

    # ---------------- FP ----------------
    X_fp_full, d_ec, d_mc = compute_fp_matrix_from_df(df_fp)
    y_fp_full      = df_fp[LABEL_COL].astype(int).to_numpy()
    smiles_fp_full = df_fp[KEY_COL].astype(str).tolist()

    # ---------------- Mordred (ALIGNED) ----------------
    md_cols        = [c for c in SHARED_MD_COLS if c in df_md.columns]
    X_md_full      = df_md[md_cols].to_numpy(dtype=np.float32)
    y_md_full      = df_md[LABEL_COL].astype(int).to_numpy()
    smiles_md_full = df_md[KEY_COL].astype(str).tolist()

    assert X_md_full.shape[1] == len(SHARED_MD_COLS), (
        f"❌ {endpoint}: MD feat_dim={X_md_full.shape[1]} "
        f"!= len(SHARED_MD_COLS)={len(SHARED_MD_COLS)}"
    )
    print(f"    MD feat_dim={X_md_full.shape[1]} ✅")

    # ---------------- Graph ----------------
    if g_obj is not None:
        graphs_full   = []
        y_g_full      = []
        smiles_g_full = []

        for item in g_obj:
            gd   = item["graph"]
            x_np = np.asarray(gd["node_features"], dtype=np.float32)

            # ✅ FIX: pastikan edge_index shape (2, E) untuk PyG
            ei_raw = np.asarray(gd["edge_index"], dtype=np.int64)
            if ei_raw.ndim == 2 and ei_raw.shape[0] != 2:
                # format (E, 2) → transpose ke (2, E)
                ei_raw = ei_raw.T
            # jika sudah (2, E), tidak perlu transpose

            graphs_full.append(
                Data(
                    x=torch.from_numpy(x_np),
                    edge_index=torch.from_numpy(ei_raw),
                )
            )
            y_g_full.append(int(item["Activity"]))
            smiles_g_full.append(item["SMILES"])

        y_g_full = np.array(y_g_full, dtype=int)
    else:
        graphs_full = y_g_full = smiles_g_full = None

    # --------- Simpan CORE_UNI per endpoint ---------
    CORE_UNI[endpoint] = {
        "fp": {"X": X_fp_full, "y": y_fp_full, "smiles": smiles_fp_full,
               "d_ec": d_ec, "d_mc": d_mc},
        "md": {"X": X_md_full, "y": y_md_full, "smiles": smiles_md_full, "cols": md_cols},
        "g":  {"graphs": graphs_full, "y": y_g_full, "smiles": smiles_g_full},
    }

    # --------- Multimodal intersection per endpoint ---------
    s_fp = set(smiles_fp_full)
    s_md = set(smiles_md_full)
    s_g  = set(smiles_g_full) if smiles_g_full is not None else None

    if s_g is not None:
        smiles_multi = sorted(list(s_fp & s_md & s_g))
    else:
        smiles_multi = sorted(list(s_fp & s_md))

    idx_fp = {s: i for i, s in enumerate(smiles_fp_full)}
    idx_md = {s: i for i, s in enumerate(smiles_md_full)}
    idx_g  = {s: i for i, s in enumerate(smiles_g_full)} if smiles_g_full else {}

    fp_arr = X_fp_full[np.array([idx_fp[s] for s in smiles_multi])]
    md_arr = X_md_full[np.array([idx_md[s] for s in smiles_multi])]
    y_arr  = y_fp_full[np.array([idx_fp[s] for s in smiles_multi])]
    gr_arr = [graphs_full[idx_g[s]] for s in smiles_multi] if graphs_full else None

    CORE_MULTI[endpoint] = {
        "X_fp": fp_arr, "X_md": md_arr, "graphs": gr_arr,
        "y": y_arr, "smiles": smiles_multi,
    }

    print(f"    multimodal size : {len(smiles_multi)}")
    print(f"    FP dim={fp_arr.shape[1]}, MD dim={md_arr.shape[1]}")

print("\n✅ CORE_UNI & CORE_MULTI built for all endpoints.")

## MTL: Joint Dataset Construction

Strategi utama MTL CardiosimTox:
- Tiap senyawa bisa punya label untuk 1, 2, atau 3 endpoint sekaligus.
- Label yang tidak tersedia di-mask (`-1`) sehingga tidak berkontribusi ke loss.
- Dataset dibentuk dari **union** SMILES semua endpoint — memaksimalkan data yang digunakan.
- Feature (FP, MD, Graph) diambil dari dataset yang menyediakan senyawa tersebut; 
  jika senyawa tidak ada di salah satu endpoint, label endpoint itu = -1 (masked).


# ========= SECTION 2 (MTL): BUILD JOINT DATASET WITH LABEL MASKING =========

In [ ]:
# ========= SECTION 2 (MTL): BUILD JOINT DATASET WITH LABEL MASKING =========

ENDPOINT_ORDER = ["hERG", "Cav1.2", "Nav1.5"]


def build_mtl_joint_dataset(core_multi_dict, endpoint_order=ENDPOINT_ORDER,
                            use_fp=True, use_md=True, use_g=False):

    smiles_per_ep = {ep: core_multi_dict[ep]["smiles"] for ep in endpoint_order}
    smiles_union  = sorted(list(set().union(*[set(v) for v in smiles_per_ep.values()])))
    N = len(smiles_union)
    smiles_to_i   = {s: i for i, s in enumerate(smiles_union)}  # lookup cepat

    print(f"[MTL] Union SMILES: {N}  (per endpoint: "
          f"{ {ep: len(v) for ep, v in smiles_per_ep.items()} })")

    # ---- Label matrix Y_mtl (N, n_ep), -1 = masked ----
    n_ep  = len(endpoint_order)
    Y_mtl = np.full((N, n_ep), fill_value=-1, dtype=np.int64)
    for j, ep in enumerate(endpoint_order):
        y_ep    = core_multi_dict[ep]["y"]
        for i_ep, smi in enumerate(smiles_per_ep[ep]):
            Y_mtl[smiles_to_i[smi], j] = int(y_ep[i_ep])   # ✅ O(N) bukan O(N²)

    # ---- Inisialisasi feature arrays ----
    X_fp_union   = None
    X_md_union   = None
    graphs_union = None

    if use_fp:
        fp_anchor = next(ep for ep in endpoint_order
                         if core_multi_dict[ep]["X_fp"] is not None)
        D_fp       = core_multi_dict[fp_anchor]["X_fp"].shape[1]
        X_fp_union = np.zeros((N, D_fp), dtype=np.float32)
        print(f"[MTL] FP anchor={fp_anchor}, D_fp={D_fp}")

    if use_md:
        md_dims = {ep: core_multi_dict[ep]["X_md"].shape[1]
                   for ep in endpoint_order
                   if core_multi_dict[ep]["X_md"] is not None}
        assert len(set(md_dims.values())) == 1, \
            f"❌ MD dims tidak konsisten: {md_dims}"
        D_md       = next(iter(md_dims.values()))
        X_md_union = np.zeros((N, D_md), dtype=np.float32)
        print(f"[MTL] MD dims per endpoint (harus sama): {md_dims}")

    if use_g:
        graphs_union = [None] * N

    # ---- Isi feature arrays — O(sum of endpoint sizes) ----
    assigned_fp = np.zeros(N, dtype=bool)
    assigned_md = np.zeros(N, dtype=bool)
    assigned_g  = np.zeros(N, dtype=bool)

    for ep in endpoint_order:
        core_ep  = core_multi_dict[ep]
        smi_list = smiles_per_ep[ep]

        for i_ep, smi in enumerate(smi_list):
            i_union = smiles_to_i[smi]

            if use_fp and not assigned_fp[i_union] and core_ep["X_fp"] is not None:
                X_fp_union[i_union]  = core_ep["X_fp"][i_ep]
                assigned_fp[i_union] = True

            if use_md and not assigned_md[i_union] and core_ep["X_md"] is not None:
                X_md_union[i_union]  = core_ep["X_md"][i_ep]
                assigned_md[i_union] = True

            if use_g and not assigned_g[i_union] and core_ep["graphs"] is not None:
                graphs_union[i_union] = core_ep["graphs"][i_ep]
                assigned_g[i_union]   = True

    # ---- Coverage report ----
    print(f"[MTL] Feature coverage: "
          f"FP={assigned_fp.sum()}/{N}, "
          f"MD={assigned_md.sum()}/{N}, "
          f"Graph={assigned_g.sum() if use_g else 'N/A'}/{N}")

    if use_fp and assigned_fp.sum() < N:
        print(f"  ⚠️  {N - assigned_fp.sum()} senyawa tidak punya FP (diisi 0)")
    if use_md and assigned_md.sum() < N:
        print(f"  ⚠️  {N - assigned_md.sum()} senyawa tidak punya MD (diisi 0)")
    if use_g and assigned_g.sum() < N:
        print(f"  ⚠️  {N - assigned_g.sum()} senyawa tidak punya Graph (None)")

    print(f"[MTL] Label coverage per endpoint:")
    for j, ep in enumerate(endpoint_order):
        n_labeled = (Y_mtl[:, j] >= 0).sum()
        n_pos     = (Y_mtl[:, j] == 1).sum()
        print(f"       {ep}: {n_labeled}/{N} labeled, "
              f"{n_pos} positives ({100*n_pos/max(n_labeled,1):.1f}%)")

    return {
        "smiles_union":  smiles_union,
        "X_fp":          X_fp_union   if use_fp else None,
        "X_md":          X_md_union   if use_md else None,
        "graphs":        graphs_union if use_g  else None,
        "Y_mtl":         Y_mtl,
        "endpoint_cols": endpoint_order,
    }


# ✅ use_g=True agar semua 7 konfigurasi bisa dijalankan
MTL_DATA = build_mtl_joint_dataset(CORE_MULTI, use_fp=True, use_md=True, use_g=True)

# ========= SECTION 3: MODEL & DATASET CLASSES (MTL) =========

In [ ]:
# ========= SECTION 3: MODEL & DATASET CLASSES (MTL) =========

try:
    from torch_geometric.data import Batch
    from torch_geometric.nn import GCNConv
    from torch_geometric.nn import global_mean_pool   # ✅ pooling yang benar
    HAS_PYG = True
except ImportError:
    Batch = GCNConv = global_mean_pool = None
    HAS_PYG = False


# ---- 3A. MTL Dataset ----
class MTLQSARDataset(Dataset):
    """
    Dataset untuk Multi-Task Learning.
    Y : np.ndarray (N, n_tasks), nilai -1 = masked.
    """
    def __init__(self, X_fp, X_md, graphs, Y):
        self.X_fp   = X_fp
        self.X_md   = X_md
        self.graphs = graphs
        self.Y      = Y
        n = len(Y)
        if X_fp   is not None: assert X_fp.shape[0] == n
        if X_md   is not None: assert X_md.shape[0] == n
        if graphs is not None: assert len(graphs)   == n

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        x_fp = torch.from_numpy(self.X_fp[idx]).float() if self.X_fp   is not None else None
        x_md = torch.from_numpy(self.X_md[idx]).float() if self.X_md   is not None else None
        g    = self.graphs[idx]                          if self.graphs is not None else None
        y    = torch.tensor(self.Y[idx], dtype=torch.long)
        return x_fp, x_md, g, y


def collate_fn_mtl(batch):
    x_fp_list, x_md_list, g_list, y_list = zip(*batch)

    has_fp = any(x is not None for x in x_fp_list)
    has_md = any(x is not None for x in x_md_list)
    has_g  = any(g is not None for g in g_list)

    x_fp_batch = torch.stack([x for x in x_fp_list], 0) if has_fp else None
    x_md_batch = torch.stack([x for x in x_md_list], 0) if has_md else None

    # ✅ FIX: filter None sebelum Batch.from_data_list
    if HAS_PYG and has_g:
        valid_graphs = [g for g in g_list if g is not None]
        g_batch = Batch.from_data_list(valid_graphs)
    else:
        g_batch = None

    y_batch = torch.stack(y_list, 0)   # (B, n_tasks)
    return x_fp_batch, x_md_batch, g_batch, y_batch


def make_mtl_loader(X_fp, X_md, graphs, Y, batch_size=64, shuffle=True):
    ds = MTLQSARDataset(X_fp, X_md, graphs, Y)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn_mtl,
    )


# ---- 3B. Encoder helpers ----
class MLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim), nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class GraphEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=128, dropout=0.0):
        super().__init__()
        if not HAS_PYG:
            raise ImportError("torch_geometric tidak tersedia")
        self.conv1   = GCNConv(in_dim, hidden_dim)
        self.conv2   = GCNConv(hidden_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, batch):
        x, ei, bidx = batch.x, batch.edge_index, batch.batch
        h = F.relu(self.conv1(x, ei))
        h = self.dropout(h)
        h = F.relu(self.conv2(h, ei))
        # ✅ FIX: pakai global_mean_pool dari PyG — lebih cepat dan tidak crash
        return global_mean_pool(h, bidx)


def _build_backbone(concat_dim, hidden_dim, n_layers, arch_type, dropout):
    arch_type = arch_type or "uniform"
    n_layers  = max(int(n_layers), 1)

    if arch_type == "uniform":
        layer_dims = [concat_dim] + [hidden_dim] * n_layers
    elif arch_type == "pyramid":
        dims = [concat_dim]; cur = hidden_dim
        for _ in range(n_layers):
            dims.append(cur); cur = max(cur // 2, 32)
        layer_dims = dims
    elif arch_type == "wide_first":
        layer_dims = [concat_dim, hidden_dim * 2] + [hidden_dim] * max(n_layers - 1, 0)
    else:
        layer_dims = [concat_dim] + [hidden_dim] * n_layers

    layers = []
    for in_d, out_d in zip(layer_dims[:-1], layer_dims[1:]):
        layers += [nn.Linear(in_d, out_d), nn.ReLU(), nn.Dropout(dropout)]
    return nn.Sequential(*layers), layer_dims[-1]


# ---- 3C. MTLModularNet ----
class MTLModularNet(nn.Module):
    """
    Shared backbone + task-specific heads.
    Forward → dict {endpoint_name: logits (B, 2)}
    """
    def __init__(
        self,
        in_fp=0, in_md=0, in_g=None,
        use_fp=True, use_md=True, use_g=False,
        hidden_dim=128, dropout=0.2,
        n_layers=2, arch_type="uniform",
        task_names=None,
    ):
        super().__init__()
        self.task_names      = task_names or ALL_ENDPOINTS
        self.safe_task_names = [t.replace(".", "_") for t in self.task_names]
        self.name_map        = {o: s for o, s in zip(self.task_names, self.safe_task_names)}

        self.use_fp = use_fp and (in_fp > 0)
        self.use_md = use_md and (in_md > 0)
        self.use_g  = use_g  and (in_g is not None) and (in_g > 0)

        fp_out = md_out = g_out = 0

        if self.use_fp:
            self.fp_encoder = MLPEncoder(in_fp, hidden_dim, hidden_dim, dropout)
            fp_out = hidden_dim
        if self.use_md:
            self.md_encoder = MLPEncoder(in_md, hidden_dim, hidden_dim, dropout)
            md_out = hidden_dim
        if self.use_g:
            self.g_encoder  = GraphEncoder(in_g, 64, hidden_dim, dropout)
            g_out  = hidden_dim

        concat_dim = fp_out + md_out + g_out
        assert concat_dim > 0, "Minimal satu modalitas harus aktif."

        self.backbone, bb_out = _build_backbone(
            concat_dim, hidden_dim, n_layers, arch_type, dropout
        )

        self.heads = nn.ModuleDict({
            safe_t: nn.Linear(bb_out, 2) for safe_t in self.safe_task_names
        })

        self.feature_dims = {
            "in_fp": in_fp, "in_md": in_md, "in_g": in_g,
            "concat_dim": concat_dim, "backbone_out": bb_out,
            "hidden_dim": hidden_dim, "n_layers": n_layers, "arch_type": arch_type,
        }

    def forward(self, x_fp=None, x_md=None, g_batch=None):
        feats = []
        if self.use_fp and x_fp    is not None: feats.append(self.fp_encoder(x_fp))
        if self.use_md and x_md    is not None: feats.append(self.md_encoder(x_md))
        if self.use_g  and g_batch is not None: feats.append(self.g_encoder(g_batch))

        assert len(feats) > 0, "Semua input None saat forward — cek modalitas aktif."
        h = feats[0] if len(feats) == 1 else torch.cat(feats, dim=1)
        z = self.backbone(h)

        return {orig: self.heads[self.name_map[orig]](z) for orig in self.task_names}


def summarize_model_mtl(model: MTLModularNet):
    n_params     = sum(p.numel() for p in model.parameters())
    n_trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("\n[MTL Model Summary]")
    print(f"  Tasks       : {model.task_names}")
    print(f"  Feature dims: {model.feature_dims}")
    print(f"  Total params: {n_params:,}")
    print(f"  Trainable   : {n_trainable:,}")
    return {"n_params": n_params, "n_trainable": n_trainable, **model.feature_dims}


print("MTL model classes defined.")

# ========= SECTION 4: MTL TRAINING LOOP, LOGGING, PLOTTING & CROSS-VALIDATION =========

In [ ]:
# ========= SECTION 4: MTL TRAINING LOOP, LOGGING, PLOTTING & CROSS-VALIDATION =========

import csv
import matplotlib.pyplot as plt

# ---- MTL TRAINING DEFAULTS ----
MAX_EPOCHS_MTL = 65
PATIENCE_MTL   = 10
GRAD_CLIP_NORM = 5.0


def compute_class_weights_mtl(Y_mtl, task_names):
    """
    Hitung class weight per task dari label matrix Y_mtl.
    Dipanggil dari Y_tr (training fold) untuk menghindari data leakage.
    Returns: dict {task_name: tensor([w0, w1])}
    """
    weights = {}
    for j, ep in enumerate(task_names):
        y_j   = Y_mtl[:, j]
        valid = y_j[y_j >= 0]
        pos   = (valid == 1).sum()
        neg   = (valid == 0).sum()
        w1 = len(valid) / (2.0 * pos) if pos > 0 else 1.0
        w0 = len(valid) / (2.0 * neg) if neg > 0 else 1.0
        weights[ep] = torch.tensor([w0, w1], dtype=torch.float32).to(DEVICE)
    return weights


def build_criterion_mtl(class_weights, task_names):
    """
    ✅ Buat CrossEntropyLoss per task SEKALI — bukan di dalam loop batch.
    Returns: dict {task_name: nn.CrossEntropyLoss}
    """
    return {
        ep: nn.CrossEntropyLoss(weight=class_weights[ep])
        for ep in task_names
    }


def mtl_loss(logits_dict, y_batch, criterions, task_names, lambda_tasks=None):
    """
    Joint MTL loss: sum of per-task weighted cross-entropy, dengan masking label -1.
    """
    if lambda_tasks is None:
        lambda_tasks = {t: 1.0 for t in task_names}

    total_loss = torch.tensor(0.0, device=DEVICE)
    n_active   = 0

    for j, ep in enumerate(task_names):
        y_j  = y_batch[:, j]
        mask = (y_j >= 0)
        if mask.sum() == 0:
            continue

        loss_j     = criterions[ep](logits_dict[ep][mask], y_j[mask])
        total_loss = total_loss + lambda_tasks[ep] * loss_j
        n_active  += 1

    return total_loss / max(n_active, 1)


def evaluate_mtl(model, loader, criterions, task_names, lambda_tasks=None):
    """
    Evaluasi model MTL pada satu DataLoader.
    Returns: (mean_loss, metrics_per_task)
    """
    model.eval()
    all_y_per_task    = {ep: [] for ep in task_names}
    all_probs_per_task = {ep: [] for ep in task_names}
    all_pred_per_task  = {ep: [] for ep in task_names}
    total_loss = 0.0
    n_samples  = 0

    with torch.no_grad():
        for x_fp, x_md, g_batch, y_batch in loader:
            y_batch = y_batch.to(DEVICE)
            if x_fp    is not None: x_fp    = x_fp.to(DEVICE,    non_blocking=NON_BLOCKING)
            if x_md    is not None: x_md    = x_md.to(DEVICE,    non_blocking=NON_BLOCKING)
            if g_batch is not None: g_batch = g_batch.to(DEVICE)

            logits_dict = model(x_fp=x_fp, x_md=x_md, g_batch=g_batch)
            loss        = mtl_loss(logits_dict, y_batch, criterions, task_names, lambda_tasks)
            total_loss += loss.item() * y_batch.size(0)
            n_samples  += y_batch.size(0)

            for j, ep in enumerate(task_names):
                y_j  = y_batch[:, j].cpu().numpy()
                mask = y_j >= 0
                if mask.sum() == 0:
                    continue
                probs = torch.softmax(logits_dict[ep], dim=1)[:, 1].cpu().numpy()
                preds = (probs > 0.5).astype(int)
                all_y_per_task[ep].append(y_j[mask])
                all_probs_per_task[ep].append(probs[mask])
                all_pred_per_task[ep].append(preds[mask])

    metrics = {}
    for ep in task_names:
        if len(all_y_per_task[ep]) == 0:
            metrics[ep] = {"auc": 0.0, "f1": 0.0, "balacc": 0.0}
            continue
        y_all    = np.concatenate(all_y_per_task[ep])
        prob_all = np.concatenate(all_probs_per_task[ep])
        pred_all = np.concatenate(all_pred_per_task[ep])
        metrics[ep] = {
            "auc":    roc_auc_score(y_all, prob_all) if len(np.unique(y_all)) > 1 else 0.5,
            "f1":     f1_score(y_all, pred_all, zero_division=0),
            "balacc": balanced_accuracy_score(y_all, pred_all),
        }

    return total_loss / max(n_samples, 1), metrics


def mean_auc_across_tasks(metrics_dict):
    aucs = [m["auc"] for m in metrics_dict.values() if m["auc"] > 0]
    return float(np.mean(aucs)) if aucs else 0.0


def append_epoch_log_mtl(out_dir, fold_id, epoch, train_loss, val_loss, mean_auc, metrics, lr_now):
    log_dir  = os.path.join(out_dir, f"fold_{fold_id}")
    os.makedirs(log_dir, exist_ok=True)
    log_csv  = os.path.join(log_dir, "epoch_log_mtl.csv")
    file_exists = os.path.exists(log_csv)

    with open(log_csv, "a", newline="") as f:
        w = csv.writer(f)
        if not file_exists:
            header = ["epoch", "train_loss", "val_loss", "mean_auc", "lr"]
            for ep in metrics:
                header += [f"{ep}_auc", f"{ep}_f1", f"{ep}_balacc"]
            w.writerow(header)
        row = [epoch, train_loss, val_loss, mean_auc, lr_now]
        for ep in metrics:
            m = metrics[ep]
            row += [m["auc"], m["f1"], m["balacc"]]
        w.writerow(row)


def plot_training_curve_mtl(csv_path, out_png_path, title=""):
    if not os.path.exists(csv_path):
        return
    df = pd.read_csv(csv_path)
    if df.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df["epoch"], df["train_loss"], marker="o", label="Train Loss")
    axes[0].plot(df["epoch"], df["val_loss"],   marker="o", label="Val Loss")
    axes[0].set_title("Loss Curve"); axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss");      axes[0].grid(True, alpha=0.3); axes[0].legend()

    if "mean_auc" in df.columns:
        axes[1].plot(df["epoch"], df["mean_auc"], marker="o", color="green", label="Mean AUC")
        axes[1].set_title("Mean AUC Curve"); axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Mean AUC");      axes[1].grid(True, alpha=0.3); axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_png_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def train_one_fold_mtl(
    model, optimizer, criterions, task_names,
    train_loader, val_loader,
    max_epochs=MAX_EPOCHS_MTL,
    patience=PATIENCE_MTL,
    config_name="", fold_id=0, out_dir="",
    lambda_tasks=None,
):
    best_state  = None
    best_auc    = 0.0
    no_improve  = 0

    # ✅ FIX: mode="max" + step(mean_auc) — lebih bersih dan benar semantiknya
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3
    )

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        n_train    = 0

        for x_fp, x_md, g_batch, y_batch in train_loader:
            y_batch = y_batch.to(DEVICE)
            if x_fp    is not None: x_fp    = x_fp.to(DEVICE,    non_blocking=NON_BLOCKING)
            if x_md    is not None: x_md    = x_md.to(DEVICE,    non_blocking=NON_BLOCKING)
            if g_batch is not None: g_batch = g_batch.to(DEVICE)

            optimizer.zero_grad()
            logits_dict = model(x_fp=x_fp, x_md=x_md, g_batch=g_batch)
            loss        = mtl_loss(logits_dict, y_batch, criterions, task_names, lambda_tasks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

            train_loss += loss.item() * y_batch.size(0)
            n_train    += y_batch.size(0)

        train_loss /= max(n_train, 1)

        val_loss, val_metrics = evaluate_mtl(
            model, val_loader, criterions, task_names, lambda_tasks
        )
        mean_auc = mean_auc_across_tasks(val_metrics)

        # ✅ FIX: step dengan mean_auc langsung (mode="max")
        scheduler.step(mean_auc)
        lr_now = optimizer.param_groups[0]["lr"]

        auc_str = " ".join(f"{ep}={val_metrics[ep]['auc']:.3f}" for ep in task_names)
        print(
            f"[MTL | {config_name} | Fold {fold_id}] "
            f"Ep {epoch:02d} | train={train_loss:.4f} val={val_loss:.4f} "
            f"| mean_AUC={mean_auc:.3f} | lr={lr_now:.2e} | {auc_str}"
        )

        append_epoch_log_mtl(
            out_dir=out_dir, fold_id=fold_id, epoch=epoch,
            train_loss=train_loss, val_loss=val_loss,
            mean_auc=mean_auc, metrics=val_metrics, lr_now=lr_now,
        )

        if mean_auc > best_auc + 1e-4:
            best_auc   = mean_auc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  -> Early stopping at epoch {epoch}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    fold_dir = os.path.join(out_dir, f"fold_{fold_id}")
    plot_training_curve_mtl(
        csv_path=os.path.join(fold_dir, "epoch_log_mtl.csv"),
        out_png_path=os.path.join(fold_dir, "training_curves_mtl.png"),
        title=f"MTL | {config_name} | Fold {fold_id}",
    )

    return model, best_auc


def run_cv_mtl(
    mtl_data,
    use_fp=True, use_md=True, use_g=False,
    n_splits=CV_FOLDS,
    batch_size=64,
    lr=1e-3,
    hidden_dim=128,
    dropout=DROPOUT,
    n_layers=2,
    arch_type="uniform",
    lambda_tasks=None,
    max_epochs=MAX_EPOCHS_MTL,
    patience=PATIENCE_MTL,
):
    cfg         = (["FP"] if use_fp else []) + (["MD"] if use_md else []) + (["Graph"] if use_g else [])
    config_name = "+".join(cfg) if cfg else "None"
    out_dir     = os.path.join(OUT_ROOT, "MTL", config_name)
    os.makedirs(out_dir, exist_ok=True)

    X_fp    = mtl_data["X_fp"]    if use_fp else None
    X_md    = mtl_data["X_md"]    if use_md else None
    graphs  = mtl_data["graphs"]  if use_g  else None
    Y_mtl   = mtl_data["Y_mtl"]
    ep_cols = mtl_data["endpoint_cols"]
    N       = len(Y_mtl)

    print(f"\n[CV MTL] Config={config_name} | N={N} | Endpoints={ep_cols}")

    in_fp = X_fp.shape[1] if X_fp is not None else 0
    in_md = X_md.shape[1] if X_md is not None else 0
    in_g  = None
    if use_g and graphs is not None:
        g0   = get_graph_example([g for g in graphs if g is not None])
        in_g = g0.x.shape[1]

    y_strat          = Y_mtl[:, ep_cols.index("hERG")].copy()
    y_strat[y_strat < 0] = 2

    skf          = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_results = []

    for fold_id, (train_idx, val_idx) in enumerate(skf.split(np.zeros(N), y_strat), 1):
        print(f"\n===== MTL | {config_name} | Fold {fold_id}/{n_splits} =====")

        Y_tr  = Y_mtl[train_idx]
        Y_val = Y_mtl[val_idx]

        X_fp_tr  = X_fp[train_idx] if X_fp is not None else None
        X_fp_val = X_fp[val_idx]   if X_fp is not None else None

        X_md_tr = X_md_val = scaler_md = None
        if X_md is not None:
            scaler_md  = StandardScaler()
            X_md_tr    = scaler_md.fit_transform(X_md[train_idx])
            X_md_val   = scaler_md.transform(X_md[val_idx])

        g_tr  = [graphs[i] for i in train_idx] if graphs is not None else None
        g_val = [graphs[i] for i in val_idx]   if graphs is not None else None

        # ✅ FIX: class_weights dihitung dari Y_tr (bukan Y_mtl penuh)
        class_weights = compute_class_weights_mtl(Y_tr, ep_cols)
        criterions    = build_criterion_mtl(class_weights, ep_cols)

        train_loader = make_mtl_loader(X_fp_tr,  X_md_tr,  g_tr,  Y_tr,  batch_size, shuffle=True)
        val_loader   = make_mtl_loader(X_fp_val, X_md_val, g_val, Y_val, batch_size, shuffle=False)

        model = MTLModularNet(
            in_fp=in_fp, in_md=in_md, in_g=in_g,
            use_fp=use_fp, use_md=use_md, use_g=use_g,
            hidden_dim=hidden_dim, dropout=dropout,
            n_layers=n_layers, arch_type=arch_type,
            task_names=ep_cols,
        ).to(DEVICE)

        summarize_model_mtl(model)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        model, best_auc = train_one_fold_mtl(
            model, optimizer, criterions, ep_cols,
            train_loader, val_loader,
            max_epochs=max_epochs, patience=patience,
            config_name=config_name, fold_id=fold_id, out_dir=out_dir,
            lambda_tasks=lambda_tasks,
        )

        fold_dir = os.path.join(out_dir, f"fold_{fold_id}")
        os.makedirs(fold_dir, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(fold_dir, "best_mtl_model.pt"))
        if scaler_md is not None:
            with open(os.path.join(fold_dir, "scaler_mordred.pkl"), "wb") as f:
                pickle.dump(scaler_md, f)

        fold_results.append({"fold": fold_id, "best_mean_auc": float(best_auc)})

    mean_auc_cv = np.mean([r["best_mean_auc"] for r in fold_results])
    print(f"[CV MTL] {config_name} Mean AUC across folds = {mean_auc_cv:.4f}")

    with open(os.path.join(out_dir, "cv_summary_mtl.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["fold", "best_mean_auc"])
        w.writeheader(); w.writerows(fold_results)

    return fold_results, mean_auc_cv


print("MTL training functions defined.")

# ========= SECTION 5: HYPERPARAMETER OPTIMIZATION (OPTUNA) FOR MTL =========

In [ ]:
# ========= SECTION 5: HYPERPARAMETER OPTIMIZATION (OPTUNA) FOR MTL =========

import optuna

N_TRIALS_MTL = 10

# ✅ Nama param Optuna harus bebas titik — mapping ke nama task asli
_LAMBDA_PARAM_MAP = {
    "hERG":   "lambda_hERG",
    "Cav1.2": "lambda_Cav1_2",   # ✅ FIX: pakai "_" bukan "."
    "Nav1.5": "lambda_Nav1_5",   # ✅ FIX: pakai "_" bukan "."
}


def _extract_lambda_tasks(params: dict) -> dict:
    """Ambil lambda_tasks dari params Optuna menggunakan _LAMBDA_PARAM_MAP."""
    return {
        task: params.get(optuna_key, 1.0)
        for task, optuna_key in _LAMBDA_PARAM_MAP.items()
    }


def make_objective_mtl(mtl_data, use_fp, use_md, use_g):
    """
    Objective function Optuna untuk MTL.
    Mengoptimasi mean AUC CV dari run_cv_mtl().
    """
    cfg = (["FP"] if use_fp else []) + (["MD"] if use_md else []) + (["Graph"] if use_g else [])
    config_name = "+".join(cfg) if cfg else "None"

    def objective(trial: optuna.Trial):
        arch_type  = trial.suggest_categorical("arch_type",  ["uniform", "pyramid", "wide_first"])
        n_layers   = trial.suggest_int("n_layers",   2, 6)
        hidden_dim = trial.suggest_int("hidden_dim", 64, 512)
        dropout    = trial.suggest_float("dropout",  0.0, 0.5)
        lr         = trial.suggest_float("lr",       1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

        # ✅ FIX: nama param Optuna tanpa titik
        lambda_hERG   = trial.suggest_float("lambda_hERG",   0.3, 3.0, log=True)
        lambda_Cav1_2 = trial.suggest_float("lambda_Cav1_2", 0.3, 3.0, log=True)
        lambda_Nav1_5 = trial.suggest_float("lambda_Nav1_5", 0.3, 3.0, log=True)

        lambda_tasks = {
            "hERG":   lambda_hERG,
            "Cav1.2": lambda_Cav1_2,
            "Nav1.5": lambda_Nav1_5,
        }

        print(
            f"\n[OPTUNA | {config_name} | Trial {trial.number}] "
            f"arch={arch_type} | layers={n_layers} | hidden={hidden_dim} "
            f"| lr={lr:.2e} | dropout={dropout:.2f} | bs={batch_size} "
            f"| lambda(hERG={lambda_hERG:.2f}, Cav1.2={lambda_Cav1_2:.2f}, "
            f"Nav1.5={lambda_Nav1_5:.2f})"
        )

        _, mean_auc = run_cv_mtl(
            mtl_data=mtl_data,
            use_fp=use_fp, use_md=use_md, use_g=use_g,
            n_splits=CV_FOLDS,
            batch_size=batch_size, lr=lr,
            hidden_dim=hidden_dim, dropout=dropout,
            n_layers=n_layers, arch_type=arch_type,
            lambda_tasks=lambda_tasks,
            max_epochs=MAX_EPOCHS_MTL,
            patience=PATIENCE_MTL,
        )

        trial.set_user_attr("mean_auc", float(mean_auc))
        return mean_auc

    return objective


def run_optuna_mtl(mtl_data, configs=None, n_trials=N_TRIALS_MTL):
    """
    Jalankan HPO Optuna untuk satu atau beberapa konfigurasi modalitas MTL.

    Parameters
    ----------
    mtl_data : dict        — output dari build_mtl_joint_dataset()
    configs  : list[dict]  — list konfigurasi modalitas; None = semua 7
    n_trials : int         — jumlah trial per konfigurasi

    Returns
    -------
    best_results : dict {config_name: {best_auc, best_params, lambda_tasks, study_name}}
    """
    if configs is None:
        configs = [
            {"name": "FP",          "use_fp": True,  "use_md": False, "use_g": False},
            {"name": "MD",          "use_fp": False,  "use_md": True,  "use_g": False},
            {"name": "Graph",       "use_fp": False, "use_md": False, "use_g": True},
            {"name": "FP+MD",       "use_fp": True,  "use_md": True,  "use_g": False},
            {"name": "FP+Graph",    "use_fp": True,  "use_md": False, "use_g": True},
            {"name": "MD+Graph",    "use_fp": False, "use_md": True,  "use_g": True},
            {"name": "FP+MD+Graph", "use_fp": True,  "use_md": True,  "use_g": True},
        ]

    master_excel = os.path.join(OUT_ROOT, "mtl_final_architectures.xlsx")
    best_results = {}

    for cfg in configs:
        config_name = cfg["name"]
        use_fp, use_md, use_g = cfg["use_fp"], cfg["use_md"], cfg["use_g"]

        print(f"\n========== [MTL HPO] {config_name} ==========")

        objective = make_objective_mtl(
            mtl_data=mtl_data, use_fp=use_fp, use_md=use_md, use_g=use_g
        )

        study = optuna.create_study(
            study_name=f"MTL_{config_name}_AUC",
            direction="maximize",
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

        best_params = study.best_params.copy()
        best_auc    = float(study.best_value)

        # ✅ FIX: pakai _extract_lambda_tasks agar key benar
        lambda_tasks = _extract_lambda_tasks(best_params)

        print(f"[BEST | {config_name}] AUC={best_auc:.4f}")
        print(f"[BEST | {config_name}] Params={best_params}")

        row = {
            "Config":          config_name,
            "BestMeanAUC_CV":  round(best_auc, 4),
            "BestParams":      str(best_params),
            "NTrials":         n_trials,
            "MaxEpochsCV":     MAX_EPOCHS_MTL,
            "PatienceCV":      PATIENCE_MTL,
            "Timestamp":       pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

        df_row = pd.DataFrame([row])
        if os.path.exists(master_excel):
            df_old = pd.read_excel(master_excel)
            df_row = pd.concat([df_old, df_row], ignore_index=True)
        df_row.to_excel(master_excel, index=False, engine="openpyxl")
        print(f"[SAVE] Best HP summary saved to: {master_excel}")

        best_results[config_name] = {
            "best_auc":    best_auc,
            "best_params": best_params,
            "lambda_tasks": lambda_tasks,
            "study_name":  f"MTL_{config_name}_AUC",
        }

    return best_results


def extract_besthp_for_final(best_hp_entry):
    """
    Ambil hyperparameter hasil HPO untuk full training.

    Returns
    -------
    best_params_clean : dict
    lambda_tasks      : dict
    """
    raw = best_hp_entry.get("best_params", best_hp_entry)

    # ✅ FIX: pakai _extract_lambda_tasks agar konsisten
    lambda_tasks = _extract_lambda_tasks(raw)

    best_params_clean = {
        "arch_type":  raw["arch_type"],
        "n_layers":   raw["n_layers"],
        "hidden_dim": raw["hidden_dim"],
        "dropout":    raw["dropout"],
        "lr":         raw["lr"],
        "batch_size": raw["batch_size"],
    }

    return best_params_clean, lambda_tasks


print(f"Default HPO trials = {N_TRIALS_MTL}.")

# ========= SECTION 6: FINAL TRAINING MTL ON FULL DEV SET =========

In [ ]:
# ========= SECTION 6: FINAL TRAINING MTL ON FULL DEV SET =========

FINAL_MAX_EPOCHS_MTL = 65


def append_final_training_log_mtl(out_dir, epoch, train_loss, lr_now):
    log_csv    = os.path.join(out_dir, "final_training_log_mtl.csv")
    file_exists = os.path.exists(log_csv)
    with open(log_csv, "a", newline="") as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(["epoch", "train_loss", "lr"])
        w.writerow([epoch, train_loss, lr_now])


def plot_final_training_curve_mtl(csv_path, out_png_path, title=""):
    if not os.path.exists(csv_path):
        return
    df = pd.read_csv(csv_path)
    if df.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df["epoch"], df["train_loss"], marker="o", label="Train Loss")
    axes[0].set_title("Final Train Loss Curve"); axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss"); axes[0].grid(True, alpha=0.3); axes[0].legend()

    if "lr" in df.columns:
        axes[1].plot(df["epoch"], df["lr"], marker="o", color="purple", label="Learning Rate")
        axes[1].set_title("Learning Rate Schedule"); axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("LR"); axes[1].grid(True, alpha=0.3); axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_png_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def train_full_mtl(
    mtl_data,
    use_fp=True, use_md=True, use_g=False,
    batch_size=64, lr=1e-3,
    hidden_dim=128, dropout=DROPOUT,
    n_layers=2, arch_type="uniform",
    lambda_tasks=None,
    max_epochs=FINAL_MAX_EPOCHS_MTL,
):
    """
    Train final MTL model pada seluruh joint dev set tanpa validation split.

    Returns
    -------
    model     : trained final model
    scaler_md : StandardScaler atau None
    """
    cfg         = (["FP"] if use_fp else []) + (["MD"] if use_md else []) + (["Graph"] if use_g else [])
    config_name = "+".join(cfg) if cfg else "None"
    out_dir     = os.path.join(OUT_ROOT, "MTL", config_name, "final")
    os.makedirs(out_dir, exist_ok=True)

    X_fp    = mtl_data["X_fp"]    if use_fp else None
    X_md    = mtl_data["X_md"]    if use_md else None
    graphs  = mtl_data["graphs"]  if use_g  else None
    Y_mtl   = mtl_data["Y_mtl"]
    ep_cols = mtl_data["endpoint_cols"]
    N       = len(Y_mtl)

    print(f"\n[FINAL MTL] Config={config_name} | N={N} | Endpoints={ep_cols}")

    if lambda_tasks is None:
        lambda_tasks = {t: 1.0 for t in ep_cols}
        print("[FINAL MTL] lambda_tasks not provided, default all = 1.0")

    # ✅ FIX: build criterions (dict of CrossEntropyLoss) — konsisten dengan Section 4
    class_weights = compute_class_weights_mtl(Y_mtl, ep_cols)
    criterions    = build_criterion_mtl(class_weights, ep_cols)

    scaler_md = None
    if X_md is not None:
        scaler_md = StandardScaler()
        X_md = scaler_md.fit_transform(X_md)

    full_loader = make_mtl_loader(
        X_fp, X_md, graphs, Y_mtl, batch_size=batch_size, shuffle=True
    )

    in_fp = X_fp.shape[1] if X_fp is not None else 0
    in_md = X_md.shape[1] if X_md is not None else 0
    in_g  = None
    if use_g and graphs is not None:
        g0   = get_graph_example([g for g in graphs if g is not None])
        in_g = g0.x.shape[1]

    model = MTLModularNet(
        in_fp=in_fp, in_md=in_md, in_g=in_g,
        use_fp=use_fp, use_md=use_md, use_g=use_g,
        hidden_dim=hidden_dim, dropout=dropout,
        n_layers=n_layers, arch_type=arch_type,
        task_names=ep_cols,
    ).to(DEVICE)

    summary = summarize_model_mtl(model)

    with open(os.path.join(out_dir, "final_mtl_architecture.txt"), "w") as f:
        f.write("FINAL MTL ARCHITECTURE\n")
        f.write(f"Config: {config_name}\n")
        f.write(
            f"Hyperparams: hidden_dim={hidden_dim}, dropout={dropout}, "
            f"n_layers={n_layers}, arch_type={arch_type}, lr={lr}, batch_size={batch_size}\n"
        )
        f.write(f"lambda_tasks: {lambda_tasks}\n")
        f.write(f"Feature dims: {model.feature_dims}\n")
        f.write(f"Total params: {summary['n_params']}\n")
        f.write(f"Trainable params: {summary['n_trainable']}\n")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max_epochs, eta_min=lr * 0.1,
    )

    # ✅ FIX: tanpa validasi, simpan last state (bukan best loss)
    # Best-loss tracking tanpa val set bisa overfit — final epoch lebih representatif
    for epoch in range(1, max_epochs + 1):
        model.train()
        epoch_loss = 0.0
        n_total    = 0

        for x_fp, x_md, g_batch, y_batch in full_loader:
            y_batch = y_batch.to(DEVICE)
            if x_fp    is not None: x_fp    = x_fp.to(DEVICE,    non_blocking=NON_BLOCKING)
            if x_md    is not None: x_md    = x_md.to(DEVICE,    non_blocking=NON_BLOCKING)
            if g_batch is not None: g_batch = g_batch.to(DEVICE)

            optimizer.zero_grad()
            logits_dict = model(x_fp=x_fp, x_md=x_md, g_batch=g_batch)
            # ✅ FIX: pakai criterions (dict), bukan class_weights
            loss = mtl_loss(logits_dict, y_batch, criterions, ep_cols, lambda_tasks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

            epoch_loss += loss.item() * y_batch.size(0)
            n_total    += y_batch.size(0)

        scheduler.step()
        epoch_loss /= max(n_total, 1)
        lr_now = optimizer.param_groups[0]["lr"]

        print(
            f"[FINAL MTL | {config_name}] Ep {epoch:02d} "
            f"| loss={epoch_loss:.4f} | lr={lr_now:.2e}"
        )
        append_final_training_log_mtl(
            out_dir=out_dir, epoch=epoch,
            train_loss=epoch_loss, lr_now=lr_now,
        )

    # ✅ FIX: pakai last state — model sudah selesai training di akhir loop
    plot_final_training_curve_mtl(
        csv_path=os.path.join(out_dir, "final_training_log_mtl.csv"),
        out_png_path=os.path.join(out_dir, "final_training_curve_mtl.png"),
        title=f"FINAL MTL | {config_name}",
    )

    torch.save(model.state_dict(), os.path.join(out_dir, "final_mtl_model.pt"))
    if scaler_md is not None:
        with open(os.path.join(out_dir, "scaler_mordred.pkl"), "wb") as f:
            pickle.dump(scaler_md, f)

    print(f"[FINAL MTL] Model saved to: {out_dir}")
    return model, scaler_md


def train_final_from_besthp(mtl_data, config, best_hp_entry, final_max_epochs=FINAL_MAX_EPOCHS_MTL):
    """Train 1 konfigurasi final model dari hasil HPO."""
    best_params, lambda_tasks = extract_besthp_for_final(best_hp_entry)

    final_model, final_scaler = train_full_mtl(
        mtl_data=mtl_data,
        use_fp=config["use_fp"], use_md=config["use_md"], use_g=config["use_g"],
        batch_size=best_params["batch_size"], lr=best_params["lr"],
        hidden_dim=best_params["hidden_dim"], dropout=best_params["dropout"],
        n_layers=best_params["n_layers"], arch_type=best_params["arch_type"],
        lambda_tasks=lambda_tasks,
        max_epochs=final_max_epochs,
    )

    return {
        "model":        final_model,
        "scaler":       final_scaler,
        "best_params":  best_params,
        "lambda_tasks": lambda_tasks,
    }


def train_all_final_from_besthp(
    mtl_data, run_configs, best_hp_all,
    final_max_epochs=FINAL_MAX_EPOCHS_MTL,
):
    """
    Train semua konfigurasi final model berdasarkan hasil HPO.

    Returns
    -------
    final_models : dict {config_name: {"model", "scaler", "best_params", "lambda_tasks"}}
    """
    final_models = {}

    for config in run_configs:
        config_name = config["name"]
        if config_name not in best_hp_all:
            print(f"[SKIP] {config_name} tidak ditemukan di best_hp_all")
            continue

        print(f"\n========== FINAL TRAIN: {config_name} ==========")
        final_models[config_name] = train_final_from_besthp(
            mtl_data=mtl_data,
            config=config,
            best_hp_entry=best_hp_all[config_name],
            final_max_epochs=final_max_epochs,
        )

    return final_models


print(f"Section 6 ready. Final full-train max epochs = {FINAL_MAX_EPOCHS_MTL}.")

# ========= SECTION 7: INFERENCE HELPER (MTL) =========

In [ ]:
# ========= SECTION 7: INFERENCE HELPER (MTL) =========

# get_graph_example sudah didefinisikan di Section 6 — tidak perlu deklarasi ulang.
# Jika Section 6 belum di-run, definisi fallback di bawah ini akan aktif.
if "get_graph_example" not in dir():
    def get_graph_example(graphs):
        """Ambil satu contoh graph yang tidak None dari list."""
        for g in graphs:
            if g is not None:
                return g
        raise ValueError("Semua graph None — tidak bisa inferensi graph.")


def predict_mtl(
    model,
    X_fp=None,
    X_md=None,
    graphs=None,
    scaler_md=None,
    batch_size=64,
):
    """
    Prediksi untuk senyawa baru menggunakan model MTL.

    Parameters
    ----------
    model      : MTLModularNet (sudah .eval() akan dipanggil di sini)
    X_fp       : np.ndarray | None  — (N, D_fp)
    X_md       : np.ndarray | None  — (N, D_md), belum di-scale
    graphs     : list | None        — (N,) PyG Data
    scaler_md  : StandardScaler | None
    batch_size : int

    Returns
    -------
    probs_dict : dict {endpoint: np.ndarray (N,)}  — probabilitas kelas 1 (blocker)
    preds_dict : dict {endpoint: np.ndarray (N,)}  — prediksi biner (0/1)
    """
    model.eval()
    task_names = list(model.task_names)

    if scaler_md is not None and X_md is not None:
        X_md = scaler_md.transform(X_md)

    if X_fp is not None:
        N = X_fp.shape[0]
    elif X_md is not None:
        N = X_md.shape[0]
    elif graphs is not None:
        N = len(graphs)
    else:
        raise ValueError("predict_mtl: semua input (X_fp, X_md, graphs) adalah None.")

    Y_dummy = np.full((N, len(task_names)), -1, dtype=np.int64)
    loader  = make_mtl_loader(
        X_fp, X_md, graphs, Y_dummy, batch_size=batch_size, shuffle=False
    )

    all_probs = {ep: [] for ep in task_names}

    with torch.no_grad():
        for x_fp_b, x_md_b, g_b, _ in loader:
            if x_fp_b is not None: x_fp_b = x_fp_b.to(DEVICE, non_blocking=NON_BLOCKING)
            if x_md_b is not None: x_md_b = x_md_b.to(DEVICE, non_blocking=NON_BLOCKING)
            if g_b    is not None: g_b    = g_b.to(DEVICE)

            logits_dict = model(x_fp=x_fp_b, x_md=x_md_b, g_batch=g_b)

            for ep in task_names:
                probs = torch.softmax(logits_dict[ep], dim=1)[:, 1].cpu().numpy()
                all_probs[ep].append(probs)

    probs_dict = {ep: np.concatenate(all_probs[ep]) for ep in task_names}
    preds_dict = {ep: (probs_dict[ep] >= 0.5).astype(int) for ep in task_names}

    return probs_dict, preds_dict


def predict_mtl_from_smiles_df(
    df,
    model,
    scaler_md=None,
    smiles_col=KEY_COL,
    fp_cols=("ECFP4", "MACCS"),
    md_feature_cols=None,
    use_fp=True,
    use_md=True,
    use_g=False,       # ✅ default False — graph harus dibangun manual sebelum inference
    graphs=None,       # ✅ FIX: tambah parameter eksplisit untuk graph input
    batch_size=64,
):
    """
    Prediksi dari DataFrame yang sudah punya kolom FP dan/atau MD.

    Parameters
    ----------
    df             : pd.DataFrame — senyawa baru
    model          : MTLModularNet
    scaler_md      : StandardScaler | None
    smiles_col     : str
    fp_cols        : tuple — nama kolom ECFP4 dan MACCS di df
    md_feature_cols: list[str] | None — kolom MD; default pakai SHARED_MD_COLS
    use_fp/md/g    : bool
    graphs         : list[PyG Data] | None — wajib diisi jika use_g=True
    batch_size     : int

    Returns
    -------
    result_df : pd.DataFrame dengan kolom SMILES + prob_{ep} + pred_{ep} per endpoint
    """
    X_fp = None
    X_md = None

    if use_fp:
        X_fp, _, _ = compute_fp_matrix_from_df(df, ec_col=fp_cols[0], mc_col=fp_cols[1])

    if use_md:
        cols         = md_feature_cols if md_feature_cols is not None else SHARED_MD_COLS
        cols_present = [c for c in cols if c in df.columns]
        if len(cols_present) < len(cols):
            print(f"[WARN] {len(cols) - len(cols_present)} MD cols tidak ada di df, diisi 0")
        X_md = np.zeros((len(df), len(cols)), dtype=np.float32)
        for i, c in enumerate(cols):
            if c in df.columns:
                X_md[:, i] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).values

    # ✅ FIX: validasi graph eksplisit — jangan diam-diam pakai graphs=None saat use_g=True
    if use_g and graphs is None:
        print(
            "[WARN] use_g=True tapi graphs=None — "
            "graph harus dibangun dari SMILES sebelum memanggil fungsi ini. "
            "Melanjutkan tanpa graph."
        )
        use_g = False

    probs_dict, preds_dict = predict_mtl(
        model=model,
        X_fp=X_fp,
        X_md=X_md,
        graphs=graphs if use_g else None,
        scaler_md=scaler_md,
        batch_size=batch_size,
    )

    result_df = df[[smiles_col]].copy().reset_index(drop=True)
    for ep in model.task_names:
        result_df[f"prob_{ep}"] = probs_dict[ep]
        result_df[f"pred_{ep}"] = preds_dict[ep]

    return result_df


def save_predictions_mtl(result_df, out_path):
    """Simpan hasil prediksi ke CSV."""
    result_df.to_csv(out_path, index=False)
    print(f"[SAVE] Predictions saved → {out_path}")


print("Section 7 (Inference Helper) ready.")

# ========= EXECUTION BLOCK: RUN HPO + FINAL TRAINING =========

In [ ]:
# ========= EXECUTION BLOCK: RUN HPO + FINAL TRAINING =========

# ----------------------------------------------------------------
# 1. KONFIGURASI EKSPERIMEN
# ----------------------------------------------------------------

RUN_CONFIGS = [
    {"name": "FP",          "use_fp": True,  "use_md": False, "use_g": False},
    # {"name": "MD",          "use_fp": False, "use_md": True,  "use_g": False},
    # {"name": "Graph",       "use_fp": False, "use_md": False, "use_g": True},
    # {"name": "FP+MD",       "use_fp": True,  "use_md": True,  "use_g": False},
    # {"name": "FP+Graph",    "use_fp": True,  "use_md": False, "use_g": True},
    # {"name": "MD+Graph",    "use_fp": False, "use_md": True,  "use_g": True},
    # {"name": "FP+MD+Graph", "use_fp": True,  "use_md": True,  "use_g": True},
]

N_TRIALS       = 10
RUN_HPO        = True
RUN_FULL_TRAIN = True
FINAL_EPOCHS   = FINAL_MAX_EPOCHS_MTL   # = 65


# ✅ FIX: inisialisasi best_hp_all sebagai dict kosong agar tidak NameError
# saat RUN_HPO=False dan RUN_FULL_TRAIN=True
best_hp_all  = {}
final_models = {}


# ----------------------------------------------------------------
# 2. JALANKAN HPO
# ----------------------------------------------------------------

if RUN_HPO:
    print("\n" + "="*60)
    print("  STAGE 1: HYPERPARAMETER OPTIMIZATION (OPTUNA)")
    print("="*60)

    best_hp_all = run_optuna_mtl(
        mtl_data=MTL_DATA,
        configs=RUN_CONFIGS,
        n_trials=N_TRIALS,
    )

    print("\n[DONE] HPO selesai. Best HP per konfigurasi:")
    for cfg_name, hp in best_hp_all.items():
        print(f"  {cfg_name}: best_auc={hp['best_auc']:.4f} | params={hp['best_params']}")

else:
    # ✅ FIX: coba load dari Excel jika tersedia, agar RUN_FULL_TRAIN tidak crash
    master_excel = os.path.join(OUT_ROOT, "mtl_final_architectures.xlsx")
    if os.path.exists(master_excel):
        print(f"[LOAD] RUN_HPO=False — memuat best_hp_all dari: {master_excel}")
        df_hp = pd.read_excel(master_excel, engine="openpyxl")
        for _, row in df_hp.iterrows():
            cfg_name = row["Config"]
            import ast
            raw_params = ast.literal_eval(row["BestParams"])
            best_hp_all[cfg_name] = {
                "best_auc":    row["BestMeanAUC_CV"],
                "best_params": raw_params,
                "lambda_tasks": _extract_lambda_tasks(raw_params),
                "study_name":  f"MTL_{cfg_name}_AUC",
            }
        print(f"  Loaded {len(best_hp_all)} konfigurasi: {list(best_hp_all.keys())}")
    else:
        print(
            f"[WARN] RUN_HPO=False dan {master_excel} tidak ditemukan. "
            "best_hp_all kosong — RUN_FULL_TRAIN akan di-skip otomatis."
        )


# ----------------------------------------------------------------
# 3. JALANKAN FULL TRAINING DARI BEST HP
# ----------------------------------------------------------------

# ✅ FIX: guard agar tidak crash jika best_hp_all kosong
if RUN_FULL_TRAIN and len(best_hp_all) > 0:
    print("\n" + "="*60)
    print("  STAGE 2: FINAL TRAINING (FULL DEV SET)")
    print("="*60)

    final_models = train_all_final_from_besthp(
        mtl_data=MTL_DATA,
        run_configs=RUN_CONFIGS,
        best_hp_all=best_hp_all,
        final_max_epochs=FINAL_EPOCHS,
    )

    print("\n[DONE] Final training selesai.")
    print(f"  Model tersimpan di: {OUT_ROOT}/MTL/<config>/final/")
    for cfg_name in final_models:
        print(f"  ✅ {cfg_name}")

elif RUN_FULL_TRAIN and len(best_hp_all) == 0:
    print(
        "[SKIP] RUN_FULL_TRAIN=True tapi best_hp_all kosong — "
        "jalankan HPO terlebih dahulu atau pastikan Excel tersedia."
    )
else:
    print("[SKIP] RUN_FULL_TRAIN=False.")


# ----------------------------------------------------------------
# 4. CONTOH INFERENSI (opsional — uncomment jika ingin test)
# ----------------------------------------------------------------

# config_to_infer = "FP+MD+Graph"
# if config_to_infer in final_models:
#     model_inf  = final_models[config_to_infer]["model"]
#     scaler_inf = final_models[config_to_infer]["scaler"]
#
#     result_df = predict_mtl_from_smiles_df(
#         df=RAW_FP["hERG"],          # ganti dengan df senyawa baru
#         model=model_inf,
#         scaler_md=scaler_inf,
#         use_fp=True,
#         use_md=True,
#         use_g=False,                # graph inferensi butuh graphs list terpisah
#     )
#
#     save_predictions_mtl(
#         result_df,
#         out_path=os.path.join(OUT_ROOT, f"predictions_{config_to_infer}.csv")
#     )
#     print(result_df.head(10))

In [ ]:
import ast


# ----------------------------------------------------------------
# 1. DEFINISI ARSITEKTUR MANUAL
# ----------------------------------------------------------------


MANUAL_HP = {
    # "MD": {
    #     "use_fp": False, "use_md": True, "use_g": False,
    #     "arch_type":  "wide_first",
    #     "n_layers":   4,
    #     "hidden_dim": 390,
    #     "dropout":    0.08821,
    #     "lr":         0.000418,
    #     "batch_size": 64,
    #     "lambda_tasks": {"hERG": 1.0, "Cav1.2": 1.0, "Nav1.5": 1.0},
    #     "best_auc_cv":  0.9231,
    # },
    # "FP+MD": {
    #     "use_fp": True, "use_md": True, "use_g": False,
    #     "arch_type":  "uniform",
    #     "n_layers":   2,
    #     "hidden_dim": 335,
    #     "dropout":    0.239746,
    #     "lr":         0.000555,
    #     "batch_size": 128,
    #     "lambda_tasks": {"hERG": 1.0, "Cav1.2": 1.0, "Nav1.5": 1.0},
    #     "best_auc_cv":  0.9444,
    # },
    "Graph": {
        "use_fp": False, "use_md": False, "use_g": True,
        "arch_type":  "wide_first",
        "n_layers":   4,
        "hidden_dim": 493,
        "dropout":    0.4685182959600666,
        "lr":         0.00018893019957356326,
        "batch_size": 64,
        "lambda_tasks": {
            "hERG":   0.7049074557337462,
            "Cav1.2": 0.3027679321726734,
            "Nav1.5": 0.9752045144911662,
        },
        # Kalau belum ada hasil CV-nya, bisa set sementara ke None atau 0.0
        "best_auc_cv":  0.0,
    },
    "FP+Graph": {
        "use_fp": True, "use_md": False, "use_g": True,
        "arch_type":  "wide_first",
        "n_layers":   4,
        "hidden_dim": 493,
        "dropout":    0.4685182959600666,
        "lr":         0.00018893019957356326,
        "batch_size": 64,
        "lambda_tasks": {
            "hERG":   0.7049074557337462,
            "Cav1.2": 0.3027679321726734,
            "Nav1.5": 0.9752045144911662,
        },
        "best_auc_cv":  0.0,
    },
}


FINAL_EPOCHS_MANUAL = FINAL_MAX_EPOCHS_MTL   # = 65 dari Section 6



# ----------------------------------------------------------------
# 2. JALANKAN FINAL TRAINING
# ----------------------------------------------------------------


final_models_manual = {}


for config_name, hp in MANUAL_HP.items():
    print(f"\n{'='*60}")
    print(f"  FINAL TRAINING MANUAL: {config_name}")
    print(f"  Best CV AUC = {hp['best_auc_cv']}")
    print(f"  Arsitektur  = {hp['arch_type']} | layers={hp['n_layers']} "
          f"| hidden={hp['hidden_dim']} | dropout={hp['dropout']:.4f} "
          f"| lr={hp['lr']:.6f} | bs={hp['batch_size']}")
    print('='*60)

    model, scaler = train_full_mtl(
        mtl_data=MTL_DATA,
        use_fp=hp["use_fp"],
        use_md=hp["use_md"],
        use_g=hp["use_g"],
        batch_size=hp["batch_size"],
        lr=hp["lr"],
        hidden_dim=hp["hidden_dim"],
        dropout=hp["dropout"],
        n_layers=hp["n_layers"],
        arch_type=hp["arch_type"],
        lambda_tasks=hp["lambda_tasks"],
        max_epochs=FINAL_EPOCHS_MANUAL,
    )

    final_models_manual[config_name] = {
        "model":        model,
        "scaler":       scaler,
        "best_params":  {k: hp[k] for k in
                         ["arch_type","n_layers","hidden_dim",
                          "dropout","lr","batch_size"]},
        "lambda_tasks": hp["lambda_tasks"],
    }

    print(f"[DONE] {config_name} selesai.")


print("\n[DONE] Semua final training manual selesai.")
print(f"  Model tersimpan di: {OUT_ROOT}/MTL/<config>/final/")
for name in final_models_manual:
    print(f"  ✅ {name}")



# ----------------------------------------------------------------
# 3. OPSIONAL: INFERENSI LANGSUNG (uncomment jika perlu)
# ----------------------------------------------------------------


# config_to_infer = "FP+Graph"
# if config_to_infer in final_models_manual:
#     model_inf  = final_models_manual[config_to_infer]["model"]
#     scaler_inf = final_models_manual[config_to_infer]["scaler"]
#
#     result_df = predict_mtl_from_smiles_df(
#         df=RAW_FP["hERG"],
#         model=model_inf,
#         scaler_md=scaler_inf,
#         use_fp=True,
#         use_md=False,
#         use_g=True,
#     )
#     save_predictions_mtl(
#         result_df,
#         out_path=os.path.join(OUT_ROOT, f"predictions_{config_to_infer}.csv")
#     )
#     print(result_df.head(10))

In [ ]:
def plot_cv_stability(out_dir, config_name, task_names, n_folds=CV_FOLDS):
    """
    Plot stabilitas training CV dalam satu gambar:
    - Baris 1 : Loss (train + val) semua fold — tampilkan mean ± std sebagai band
    - Baris 2 : AUC per endpoint semua fold — tampilkan mean ± std sebagai band
    """
    # ---- Kumpulkan data dari semua fold ----
    fold_train_loss = []
    fold_val_loss   = []
    fold_mean_auc   = []
    fold_ep_auc     = {ep: [] for ep in task_names}

    max_epoch = 0
    for fold_id in range(1, n_folds + 1):
        csv_path = os.path.join(out_dir, f"fold_{fold_id}", "epoch_log_mtl.csv")
        if not os.path.exists(csv_path):
            continue
        df = pd.read_csv(csv_path)
        fold_train_loss.append(df["train_loss"].values)
        fold_val_loss.append(df["val_loss"].values)
        fold_mean_auc.append(df["mean_auc"].values)
        for ep in task_names:
            col = f"{ep}_auc"
            if col in df.columns:
                fold_ep_auc[ep].append(df[col].values)
        max_epoch = max(max_epoch, len(df))

    if max_epoch == 0:
        print(f"[SKIP] Tidak ada epoch log ditemukan di {out_dir}")
        return

    # ---- Pad semua array ke panjang max_epoch ----
    def pad_and_stack(arrays):
        padded = [
            np.pad(a, (0, max_epoch - len(a)), mode="edge") for a in arrays
        ]
        mat = np.stack(padded)           # (n_folds, max_epoch)
        return mat.mean(0), mat.std(0)   # mean, std per epoch

    epochs = np.arange(1, max_epoch + 1)
    n_rows = 1 + len(task_names)         # baris 1: loss+mean_auc | baris 2+: per-endpoint AUC
    fig, axes = plt.subplots(n_rows, 2, figsize=(14, 4 * n_rows))

    # ---- Baris 0 kiri: Loss curve ----
    ax = axes[0, 0]
    for label, fold_data, color in [
        ("Train Loss", fold_train_loss, "steelblue"),
        ("Val Loss",   fold_val_loss,   "tomato"),
    ]:
        m, s = pad_and_stack(fold_data)
        ax.plot(epochs, m, color=color, label=f"{label} (mean)", lw=2)
        ax.fill_between(epochs, m - s, m + s, alpha=0.2, color=color, label=f"± 1 std")
    ax.set_title("Loss Stability (all folds)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

    # ---- Baris 0 kanan: Mean AUC curve ----
    ax = axes[0, 1]
    m, s = pad_and_stack(fold_mean_auc)
    ax.plot(epochs, m, color="green", label="Mean AUC (mean)", lw=2)
    ax.fill_between(epochs, m - s, m + s, alpha=0.2, color="green", label="± 1 std")
    # Plot setiap fold sebagai garis tipis untuk melihat variasi
    for i, fa in enumerate(fold_mean_auc):
        padded = np.pad(fa, (0, max_epoch - len(fa)), mode="edge")
        ax.plot(epochs, padded, alpha=0.25, lw=0.8, color="gray")
    ax.set_title("Mean AUC Stability (all folds)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Mean AUC")
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

    # ---- Baris 1+: AUC per endpoint ----
    colors_ep = ["royalblue", "darkorange", "mediumseagreen"]
    for row, ep in enumerate(task_names, start=1):
        data = fold_ep_auc[ep]
        if not data:
            continue
        color = colors_ep[row - 1 % len(colors_ep)]

        # Kiri: mean ± std band
        ax = axes[row, 0]
        m, s = pad_and_stack(data)
        ax.plot(epochs, m, color=color, label=f"{ep} AUC (mean)", lw=2)
        ax.fill_between(epochs, m - s, m + s, alpha=0.2, color=color, label="± 1 std")
        ax.set_title(f"{ep} — AUC Stability")
        ax.set_xlabel("Epoch"); ax.set_ylabel("AUC")
        ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

        # Kanan: setiap fold individual
        ax = axes[row, 1]
        for i, fa in enumerate(data):
            padded = np.pad(fa, (0, max_epoch - len(fa)), mode="edge")
            ax.plot(epochs, padded, alpha=0.5, lw=1.0, label=f"Fold {i+1}")
        ax.set_title(f"{ep} — AUC per Fold")
        ax.set_xlabel("Epoch"); ax.set_ylabel("AUC")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=6, ncol=2)

    fig.suptitle(f"Training Stability — {config_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()

    out_png = os.path.join(out_dir, "cv_stability_plot.png")
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[PLOT] Stability plot saved → {out_png}")